# Савкина Мария, БД-251м
## Экзаменационное задание

## 1. Подготовка окружения

In [1]:
# Установка необходимых библиотек
!pip install pyspark matplotlib seaborn pandas numpy

In [2]:
# Импорт библиотек
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, avg, stddev, hour, to_timestamp,
    regexp_extract, when, lit, window, desc, asc,
    sum as spark_sum,round as spark_round, rand, randn, least,
    date_format
)

from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, TimestampType, DoubleType
)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Настройка визуализации
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

from pyspark.ml.feature import StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import OneHotEncoder
from pyspark.ml.feature import VectorAssembler, StandardScaler, FeatureHasher

## 2. Инициализация Spark Session

In [3]:
# Создание SparkSession
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("pw01_var25") \
    .master("local[*]") \
    .config("root.hadoop.fs.defaultFS", "hdfs://localhost:9000") \
    .config("spark.ui.port", "4040") \
    .config("spark.sql.shuffle.partitions", "300") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "3g") \
    .getOrCreate()

print("SparkSession успешно запущен с расширенными настройками памяти.")

# Установка уровня логирования
spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"Spark UI: http://localhost:4040")

26/07/01 00:12:45 WARN Utils: Your hostname, devopsvm resolves to a loopback address: 127.0.1.1; using 192.168.0.137 instead (on interface enp0s3)
26/07/01 00:12:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/01 00:12:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession успешно запущен с расширенными настройками памяти.
Spark Version: 3.5.3
Spark UI: http://localhost:4040


## 3. Загрузка данных

Выполнить команды из п.3 файла README.md


In [ ]:
# Загрузка данных из HDFS в Spark DataFrame
hdfs_path = "hdfs://localhost:9000/user/hadoop/task11/input/digits.csv"

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(hdfs_path)

print("Схема данных:")
df.printSchema()

print("Общая статистика:")
print(f"Всего записей: {df.count():,}")

print("\nПримеры данных:")
df.show(5, truncate=False)

Схема данных:
root
 |-- _c0: integer (nullable = true)
 |-- borough1: integer (nullable = true)
 |-- neighborhood: string (nullable = true)
 |-- building_class_category: string (nullable = true)
 |-- tax_class: string (nullable = true)
 |-- block: integer (nullable = true)
 |-- lot: integer (nullable = true)
 |-- easement: string (nullable = true)
 |-- building_class: string (nullable = true)
 |-- address9: string (nullable = true)
 |-- apartment_number: string (nullable = true)
 |-- zip_code: integer (nullable = true)
 |-- residential_units: integer (nullable = true)
 |-- commercial_units: integer (nullable = true)
 |-- total_units: integer (nullable = true)
 |-- land_sqft: double (nullable = true)
 |-- gross_sqft: double (nullable = true)
 |-- year_built: integer (nullable = true)
 |-- tax_class_at_sale: integer (nullable = true)
 |-- building_class_at_sale: string (nullable = true)
 |-- sale_price: double (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- year_of_sale: in

Всего записей: 390,883

Примеры данных:


26/07/01 00:12:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---+--------+---------------------+-----------------------+---------+-----+----+--------+--------------+-------------------+----------------+--------+-----------------+----------------+-----------+---------+----------+----------+-----------------+----------------------+------------+----------+------------+---------+---+------+------+----------+-------+-------+--------+----------+----------+----------+---------+----------+--------+-------------------+---------+---------+---------+---------+--------+--------+-------+-------+-------+---------+---------+---------+-------+---------+---------+---------------------+-------+--------+-------+-------+----------+----------+----------+---------+----------+---------+----------+--------+---------+--------+----------+--------+--------+---------+---------+---+--------+----------+-------+--------+----------+---------+----------+---------+---------+----------+----------+----------------------------------+--------+--------+--------+-------+--------+----

26/07/01 00:12:58 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , borough, neighborhood, building_class_category, tax_class, block, lot, easement, building_class, address, apartment_number, zip_code, residential_units, commercial_units, total_units, land_sqft, gross_sqft, year_built, tax_class_at_sale, building_class_at_sale, sale_price, sale_date, year_of_sale, Borough, CD, CT2010, CB2010, SchoolDist, Council, ZipCode, FireComp, PolicePrct, HealthCent, HealthArea, SanitBoro, SanitDistr, SanitSub, Address, ZoneDist1, ZoneDist2, ZoneDist3, ZoneDist4, Overlay1, Overlay2, SPDist1, SPDist2, SPDist3, LtdHeight, SplitZone, BldgClass, LandUse, Easements, OwnerType, OwnerName, LotArea, BldgArea, ComArea, ResArea, OfficeArea, RetailArea, GarageArea, StrgeArea, FactryArea, OtherArea, AreaSource, NumBldgs, NumFloors, UnitsRes, UnitsTotal, LotFront, LotDepth, BldgFront, BldgDepth, Ext, ProxCode, IrrLotCode, LotType, BsmtCode, AssessLand, AssessTot, ExemptLand, ExemptTo

## 4. Очистка данных (обработка NULL, дубликатов)

In [ ]:
# Удаление дубликатов
df = df.dropDuplicates()

# Удаляем строки с критическими пропусками (пропущена целевая метка)
df = df.dropna(subset=["label"])

# Пропущенные значения пикселей заполним нулями
pixel_cols = [f"pixel_{i}" for i in range(64)]
fill_values = {col_name: 0.0 for col_name in pixel_cols}

df = df.fillna(fill_values)


# Явное приведение типов

df = df.withColumn("label", col("label").cast(IntegerType()))

for col_name in pixel_cols:
    df = df.withColumn(col_name, col(col_name).cast(DoubleType()))

print("Количество строк после очистки:", df.count())

26/07/01 00:12:59 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , borough, neighborhood, building_class_category, tax_class, block, lot, easement, building_class, address, apartment_number, zip_code, residential_units, commercial_units, total_units, land_sqft, gross_sqft, year_built, tax_class_at_sale, building_class_at_sale, sale_price, sale_date, year_of_sale, Borough, CD, CT2010, CB2010, SchoolDist, Council, ZipCode, FireComp, PolicePrct, HealthCent, HealthArea, SanitBoro, SanitDistr, SanitSub, Address, ZoneDist1, ZoneDist2, ZoneDist3, ZoneDist4, Overlay1, Overlay2, SPDist1, SPDist2, SPDist3, LtdHeight, SplitZone, BldgClass, LandUse, Easements, OwnerType, OwnerName, LotArea, BldgArea, ComArea, ResArea, OfficeArea, RetailArea, GarageArea, StrgeArea, FactryArea, OtherArea, AreaSource, NumBldgs, NumFloors, UnitsRes, UnitsTotal, LotFront, LotDepth, BldgFront, BldgDepth, Ext, ProxCode, IrrLotCode, LotType, BsmtCode, AssessLand, AssessTot, ExemptLand, ExemptTo

Количество строк после очистки: 390883


## 5. Кластеризация

In [ ]:
#Сборка признаков в единый вектор

feature_columns = [c for c in df.columns if c != "label"]

assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")

data = assembler.transform(df)

data.select("features").show(5, truncate=False)

26/07/01 00:13:12 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , borough, neighborhood, building_class_category, tax_class, block, lot, easement, building_class, address, apartment_number, zip_code, residential_units, commercial_units, total_units, land_sqft, gross_sqft, year_built, tax_class_at_sale, building_class_at_sale, sale_price, sale_date, year_of_sale, Borough, CD, CT2010, CB2010, SchoolDist, Council, ZipCode, FireComp, PolicePrct, HealthCent, HealthArea, SanitBoro, SanitDistr, SanitSub, Address, ZoneDist1, ZoneDist2, ZoneDist3, ZoneDist4, Overlay1, Overlay2, SPDist1, SPDist2, SPDist3, LtdHeight, SplitZone, BldgClass, LandUse, Easements, OwnerType, OwnerName, LotArea, BldgArea, ComArea, ResArea, OfficeArea, RetailArea, GarageArea, StrgeArea, FactryArea, OtherArea, AreaSource, NumBldgs, NumFloors, UnitsRes, UnitsTotal, LotFront, LotDepth, BldgFront, BldgDepth, Ext, ProxCode, IrrLotCode, LotType, BsmtCode, AssessLand, AssessTot, ExemptLand, ExemptTo

медиана цены: 560000.0


In [ ]:
#Масштабирование признаков
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures", withMean=True, withStd=True)

scaler_model = scaler.fit(data)

scaled_data = scaler_model.transform(data)

scaled_data.select("scaledFeatures").show(5, truncate=False)

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml.feature import PCA
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType

#Поиск оптимального количества кластеров (k)
ks = list(range(2, 12)) #от 2 до 11

costs = []
silhouettes = []

evaluator = ClusteringEvaluator(featuresCol="scaledFeatures", predictionCol="prediction", metricName="silhouette")

for k in ks:

    kmeans = KMeans(featuresCol="scaledFeatures", predictionCol="prediction", k=k, seed=42)
    model = kmeans.fit(scaled_data)
    predictions = model.transform(scaled_data)

    silhouettes.append(evaluator.evaluate(predictions))

    costs.append(model.summary.trainingCost)


In [ ]:
#Силуэт
plt.figure(figsize=(8,5))

plt.plot(
    ks,
    silhouettes,
    marker='o'
)

plt.xlabel("Количество кластеров")
plt.ylabel("Silhouette")

plt.title("Silhouette Score")

plt.show()

best_k = ks[silhouettes.index(max(silhouettes))]

print("Лучшее k =", best_k)

In [ ]:
#Обучение финальной модели
kmeans = KMeans(
    featuresCol="scaledFeatures",
    predictionCol="prediction",
    k=best_k,
    seed=42
)

model = kmeans.fit(scaled_data)

clustered = model.transform(scaled_data)

clustered.select("label", "prediction").show(10)

In [ ]:
# Понижение размерности до двух компонент
pca = PCA(
    k=2,
    inputCol="scaledFeatures",
    outputCol="pcaFeatures"
)

pca_model = pca.fit(clustered)

pca_df = pca_model.transform(clustered)

# Извлечение координат X и Y из вектора PCA
extract_x = udf(lambda v: float(v[0]), FloatType())
extract_y = udf(lambda v: float(v[1]), FloatType())

final_export_df = (
    pca_df
    .withColumn("pca_x", extract_x("pcaFeatures"))
    .withColumn("pca_y", extract_y("pcaFeatures"))
    .select("label", "prediction", "pca_x", "pca_y")
)

final_export_df.show(10)

In [ ]:
#DataLens
import os



local_pdf = final_export_df.toPandas()

output_path = "model_results_for_datalens11.csv"

local_pdf.to_csv(output_path, index=False, encoding='utf-8')

print(f"Файл успешно сохранен локально.")
print(f"Имя файла для скачивания: {os.path.abspath(output_path)}")
print(f"Всего строк для выгрузки в DataLens: {len(local_pdf)}")

26/07/01 00:13:57 WARN CSVHeaderChecker: CSV header does not conform to the schema.
 Header: , borough, neighborhood, building_class_category, tax_class, block, lot, easement, building_class, address, apartment_number, zip_code, residential_units, commercial_units, total_units, land_sqft, gross_sqft, year_built, tax_class_at_sale, building_class_at_sale, sale_price, sale_date, year_of_sale, Borough, CD, CT2010, CB2010, SchoolDist, Council, ZipCode, FireComp, PolicePrct, HealthCent, HealthArea, SanitBoro, SanitDistr, SanitSub, Address, ZoneDist1, ZoneDist2, ZoneDist3, ZoneDist4, Overlay1, Overlay2, SPDist1, SPDist2, SPDist3, LtdHeight, SplitZone, BldgClass, LandUse, Easements, OwnerType, OwnerName, LotArea, BldgArea, ComArea, ResArea, OfficeArea, RetailArea, GarageArea, StrgeArea, FactryArea, OtherArea, AreaSource, NumBldgs, NumFloors, UnitsRes, UnitsTotal, LotFront, LotDepth, BldgFront, BldgDepth, Ext, ProxCode, IrrLotCode, LotType, BsmtCode, AssessLand, AssessTot, ExemptLand, ExemptTo

Файл успешно сохранен локально.
Имя файла для скачивания: /home/devops/Downloads/model_results_for_datalens.csv
Всего строк для выгрузки в DataLens: 28863


#Остановка Spark

In [14]:
# Остановка SparkSession
spark.stop()
print("SparkSession остановлен")

SparkSession остановлен
